# 🌿 Détection de Maladies des Plantes — Deep Learning (PlantVillage)

**Projet complet, version pro, prêt à exécuter sur Google Colab.**

Pipeline : `Téléchargement → Exploration → Prétraitement → Augmentation → 2 Modèles (EfficientNetB0 vs ResNet50V2) → Fine-Tuning → Comparaison → Évaluation → Sauvegarde → Déploiement Gradio`

---

### ⚡ Comment utiliser ce notebook

**Première fois (entraînement) :** Exécute les blocs **1 → 13** dans l'ordre (`Exécution ▸ Tout exécuter`). Le meilleur modèle est sauvegardé sur ton Google Drive.

**Les fois suivantes (juste tester / déployer) :** Tu n'as PAS besoin de ré-entraîner. Exécute seulement :
- **Bloc 1** (installation)
- **Bloc 2** (imports)
- **Bloc 4** (configuration)
- **Bloc 14** (charger le modèle sauvegardé)
- **Bloc 15** (interface Gradio)

> ✅ Active le GPU : `Exécution ▸ Modifier le type d'exécution ▸ GPU (T4)`


## 🧱 BLOC 1 — Installation des dépendances

In [ ]:
# Colab a déjà TensorFlow. On installe juste ce qui manque.
!pip install -q kagglehub gradio
print("Dépendances installées.")

## 📦 BLOC 2 — Imports & vérification du GPU

In [ ]:
import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, roc_auc_score, cohen_kappa_score)

# Reproductibilité
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU disponible :", bool(gpus))
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("Memory growth :", e)
print("✅ Imports OK")

## 📥 BLOC 3 — Téléchargement du dataset (PlantVillage via Kaggle)

In [ ]:
import kagglehub

# Télécharge le dataset PlantVillage (15 classes : Tomate, Poivron, Pomme de terre)
download_path = kagglehub.dataset_download("emmarex/plantdisease")
print("Téléchargé dans :", download_path)

# Détection automatique du dossier qui contient les sous-dossiers de classes
DATASET_DIR = None
best_count = 0
for root, dirs, files in os.walk(download_path):
    # On cherche le dossier qui contient le plus de sous-dossiers (= les classes)
    subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
    if len(subdirs) > best_count:
        best_count = len(subdirs)
        DATASET_DIR = root

print("✅ Dossier des classes détecté :", DATASET_DIR)
print("Nombre de classes trouvées   :", best_count)
print("Exemples de classes          :", sorted(os.listdir(DATASET_DIR))[:5])

## ⚙️ BLOC 4 — Configuration & hyperparamètres

In [ ]:
# --- Paramètres image / entraînement ---
IMG_SIZE   = 224          # taille native d'EfficientNetB0 / ResNet50V2
BATCH_SIZE = 32
VAL_SPLIT  = 0.30         # 30% pour (validation + test), 70% pour l'entraînement

EPOCHS_PHASE1 = 10        # entraînement de la tête (base gelée)
EPOCHS_PHASE2 = 8         # fine-tuning (base dégelée)

LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5          # 100x plus petit pour le fine-tuning

# --- Chemin de sauvegarde sur Google Drive (persiste entre les sessions) ---
DRIVE_DIR  = "/content/drive/MyDrive/PlantDisease_DL"
MODEL_PATH = os.path.join(DRIVE_DIR, "best_model.keras")
CLASSES_JSON = os.path.join(DRIVE_DIR, "class_names.json")

AUTOTUNE = tf.data.AUTOTUNE
print("Configuration prête. IMG_SIZE =", IMG_SIZE, "| BATCH_SIZE =", BATCH_SIZE)

## 🔎 BLOC 5 — Exploration du dataset

In [ ]:
# Compter les images par classe
class_dirs = sorted([d for d in os.listdir(DATASET_DIR)
                     if os.path.isdir(os.path.join(DATASET_DIR, d))])
counts = {}
for c in class_dirs:
    p = os.path.join(DATASET_DIR, c)
    n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    counts[c] = n

total = sum(counts.values())
print("Nombre total d'images :", total)
print("Nombre de classes     :", len(class_dirs))

# Graphique de distribution
plt.figure(figsize=(11, 6))
plt.barh(list(counts.keys()), list(counts.values()), color="forestgreen")
plt.xlabel("Nombre d'images")
plt.title("Distribution des images par classe")
plt.tight_layout()
plt.show()

In [ ]:
# Afficher un échantillon d'image par classe
import cv2
n = len(class_dirs)
cols = 5
rows = int(np.ceil(n / cols))
plt.figure(figsize=(15, 3 * rows))
for i, c in enumerate(class_dirs):
    p = os.path.join(DATASET_DIR, c)
    f = [x for x in os.listdir(p) if x.lower().endswith(('.jpg', '.jpeg', '.png'))][0]
    img = cv2.cvtColor(cv2.imread(os.path.join(p, f)), cv2.COLOR_BGR2RGB)
    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(img)
    ax.set_title(c[:28], fontsize=8)
    ax.axis("off")
plt.suptitle("Un échantillon par classe", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 🗂️ BLOC 6 — Chargement & découpage stratifié Train / Validation / Test

On liste tous les fichiers image, puis on fait un **découpage stratifié 70% / 15% / 15%** avec `train_test_split(stratify=...)` : **chaque classe est représentée proportionnellement dans les trois jeux** (indispensable pour une évaluation honnête sur les 15 classes).

Les images sont lues depuis le disque à la volée via `tf.data` (aucune saturation de RAM). Les pixels restent dans `[0,255]` ; la normalisation propre à chaque architecture est faite **à l'intérieur du modèle**.

In [ ]:
import pathlib
from sklearn.model_selection import train_test_split

# Lister tous les fichiers image et leur classe
data_dir = pathlib.Path(DATASET_DIR)
CLASS_NAMES = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
NB_CLASSES = len(CLASS_NAMES)

file_paths, labels = [], []
for idx, c in enumerate(CLASS_NAMES):
    for f in (data_dir / c).iterdir():
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            file_paths.append(str(f)); labels.append(idx)
file_paths = np.array(file_paths); labels = np.array(labels)
print("Total images :", len(file_paths), "| classes :", NB_CLASSES)

# Découpage stratifié 70% / 15% / 15%
trainval_f, test_f, trainval_l, test_l = train_test_split(
    file_paths, labels, test_size=0.15, stratify=labels, random_state=SEED)
train_f, val_f, train_l, val_l = train_test_split(
    trainval_f, trainval_l, test_size=0.1765, stratify=trainval_l, random_state=SEED)
print("Train :", len(train_f), "| Val :", len(val_f), "| Test :", len(test_f))

# Pipeline tf.data : lecture + redimensionnement (pixels gardés en [0,255])
def _load(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, tf.one_hot(label, NB_CLASSES)

def make_ds(files, labs, training):
    ds = tf.data.Dataset.from_tensor_slices((files, labs))
    if training:
        ds = ds.shuffle(len(files), seed=SEED)
    return ds.map(_load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)

train_ds = make_ds(train_f, train_l, training=True)
val_ds   = make_ds(val_f,   val_l,   training=False)
test_ds  = make_ds(test_f,  test_l,  training=False)

# Vérification : toutes les classes sont présentes partout
for name, labs in [("Train", train_l), ("Val", val_l), ("Test", test_l)]:
    print(name, "-> nb classes :", len(set(labs.tolist())))
print("Classes :", NB_CLASSES, "| Noms :", CLASS_NAMES)

## 🎨 BLOC 7 — Augmentation des données & pipeline de performance

L'augmentation est appliquée **uniquement à l'entraînement** (pas en validation/test, pas au déploiement). Les images restent dans l'intervalle `[0, 255]` ; la normalisation propre à chaque architecture est faite **à l'intérieur du modèle**.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")

# Appliquer l'augmentation au train uniquement, + prefetch pour la vitesse
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                        num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds  = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Visualisation de l'augmentation sur une image
for images, _ in train_ds.take(1):
    plt.figure(figsize=(10, 6))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.axis("off")
    plt.suptitle("Exemples d'augmentation (batch d'entraînement)", fontweight="bold")
    plt.show()
    break

## 🏗️ BLOC 8 — Constructeur de modèle (Transfer Learning + tête commune)

Une seule fonction construit **les deux modèles**. La tête de classification est identique ; seule la base pré-entraînée change.

**Tête :** `GlobalAveragePooling2D → BatchNorm → Dropout(0.4) → Dense(256, swish) → BatchNorm → Dropout(0.3) → Dense(NB_CLASSES, softmax)`

> Astuce pro : EfficientNet attend des pixels `[0,255]` (sa normalisation est interne).
> ResNet50V2 attend `[-1,1]` → on ajoute une couche `Rescaling` (sérialisable, contrairement à `Lambda`).

In [ ]:
def make_model(name, base_cls, needs_rescaling):
    base = base_cls(include_top=False, weights="imagenet",
                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False  # gelée pour la phase 1

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = inputs
    if needs_rescaling:
        # ResNet50V2 : [0,255] -> [-1,1]
        x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)
    # training=False garde les BatchNorm de la base en mode inférence (recommandé)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NB_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=name)
    return model, base


def train_two_phases(model, base, tag):
    cbs = [
        EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ]

    # --- Phase 1 : tête seulement ---
    print("\n=== [%s] Phase 1 : entraînement de la tête ===" % tag)
    model.compile(optimizer=keras.optimizers.Adam(LR_PHASE1),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds,
                   epochs=EPOCHS_PHASE1, callbacks=cbs, verbose=1)

    # --- Phase 2 : fine-tuning (base dégelée, LR très faible) ---
    print("\n=== [%s] Phase 2 : fine-tuning ===" % tag)
    base.trainable = True
    model.compile(optimizer=keras.optimizers.Adam(LR_PHASE2),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
                  metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds,
                   epochs=EPOCHS_PHASE2, callbacks=cbs, verbose=1)

    # Fusionner les historiques
    hist = {}
    for k in h1.history:
        hist[k] = h1.history[k] + h2.history.get(k, [])
    return hist

print("Fonctions prêtes.")

## 🤖 BLOC 9 — Modèle 1 : EfficientNetB0 (entraînement + fine-tuning)

In [ ]:
import time
t0 = time.time()
eff_model, eff_base = make_model("EfficientNetB0", EfficientNetB0, needs_rescaling=False)
eff_history = train_two_phases(eff_model, eff_base, "EfficientNetB0")
print("⏱️ EfficientNetB0 entraîné en %.1f min" % ((time.time() - t0) / 60))

## 🤖 BLOC 10 — Modèle 2 : ResNet50V2 (entraînement + fine-tuning)

In [ ]:
t0 = time.time()
res_model, res_base = make_model("ResNet50V2", ResNet50V2, needs_rescaling=True)
res_history = train_two_phases(res_model, res_base, "ResNet50V2")
print("⏱️ ResNet50V2 entraîné en %.1f min" % ((time.time() - t0) / 60))

## 🏆 BLOC 11 — Comparaison des deux modèles & sélection du meilleur

In [ ]:
# Évaluer sur le jeu de test
eff_loss, eff_acc = eff_model.evaluate(test_ds, verbose=0)
res_loss, res_acc = res_model.evaluate(test_ds, verbose=0)

print("EfficientNetB0  -> accuracy test : %.4f" % eff_acc)
print("ResNet50V2      -> accuracy test : %.4f" % res_acc)

if eff_acc >= res_acc:
    best_model, best_name, best_history = eff_model, "EfficientNetB0", eff_history
else:
    best_model, best_name, best_history = res_model, "ResNet50V2", res_history
print("\n✅ Meilleur modèle :", best_name)

# Courbes d'apprentissage des deux modèles
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for hist, lbl in [(eff_history, "EfficientNetB0"), (res_history, "ResNet50V2")]:
    axes[0].plot(hist["val_accuracy"], label=lbl)
    axes[1].plot(hist["val_loss"], label=lbl)
axes[0].set_title("Validation Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.show()

## 📊 BLOC 12 — Évaluation détaillée du meilleur modèle

In [ ]:
import numpy as np

# Vraies étiquettes et prédictions (test_ds non mélangé -> ordre stable)
y_true_oh = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_score   = best_model.predict(test_ds, verbose=0)
y_true = y_true_oh.argmax(axis=1)
y_pred = y_score.argmax(axis=1)

# --- Matrice de confusion (les 15 classes) ---
cm = confusion_matrix(y_true, y_pred, labels=range(NB_CLASSES))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Matrice de confusion — %s" % best_name)
plt.ylabel("Vraie classe"); plt.xlabel("Classe prédite")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# --- Rapport de classification ---
print("Rapport de classification :\n")
print(classification_report(y_true, y_pred, labels=range(NB_CLASSES),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))

# --- Métriques globales ---
print("Accuracy globale  : %.4f" % (y_true == y_pred).mean())
print("Kappa de Cohen    : %.4f" % cohen_kappa_score(y_true, y_pred))
print("AUC macro (OvR)   : %.4f" % roc_auc_score(y_true_oh, y_score,
                                                  average="macro", multi_class="ovr"))

In [ ]:
# --- Courbes ROC par classe (One-vs-Rest) ---
plt.figure(figsize=(10, 8))
aucs = []
for i in range(NB_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_oh[:, i], y_score[:, i])
    a = auc(fpr, tpr)
    aucs.append(a)
    plt.plot(fpr, tpr, lw=1, label="%s (AUC=%.2f)" % (CLASS_NAMES[i][:18], a))
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("Taux de faux positifs"); plt.ylabel("Taux de vrais positifs")
plt.title("Courbes ROC par classe — AUC moyen = %.3f" % np.mean(aucs))
plt.legend(loc="lower right", fontsize=7)
plt.show()

## 💾 BLOC 13 — Sauvegarde du meilleur modèle sur Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

os.makedirs(DRIVE_DIR, exist_ok=True)
best_model.save(MODEL_PATH)
with open(CLASSES_JSON, "w") as f:
    json.dump(CLASS_NAMES, f)

print("✅ Modèle sauvegardé :", MODEL_PATH)
print("✅ Classes sauvegardées :", CLASSES_JSON)
print("Tu peux maintenant fermer/relancer Colab et aller directement aux blocs 14 et 15.")

## 🚀 BLOC 14 — Charger le modèle sauvegardé (déploiement)

**Bloc indépendant.** Après un redémarrage, exécute les blocs **1, 2, 4** puis ce bloc-ci pour récupérer ton modèle sans ré-entraîner.

In [ ]:
from google.colab import drive
import os, json
import tensorflow as tf

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

model = tf.keras.models.load_model(MODEL_PATH)
with open(CLASSES_JSON) as f:
    CLASS_NAMES = json.load(f)
NB_CLASSES = len(CLASS_NAMES)

print("✅ Modèle chargé. Classes :", NB_CLASSES)
print(CLASS_NAMES)

## 🖥️ BLOC 15 — Interface Gradio (test en ligne)

Charge une photo de feuille → le modèle prédit le **type de plante + l'état + la maladie** avec un **score de confiance**. Un lien public est généré (`share=True`).

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf

def pretty(label):
    # 'Tomato_Early_blight' -> 'Tomato — Early blight'
    parts = label.replace("___", "_").replace("__", "_").split("_")
    plant = parts[0]
    rest = " ".join(parts[1:]).strip() if len(parts) > 1 else ""
    return plant, rest

def predict(img):
    if img is None:
        return {}, "Charge une image."
    x = tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))
    x = tf.expand_dims(x, 0)                       # le modèle gère [0,255] -> normalisation interne
    probs = model.predict(x, verbose=0)[0]
    confidences = {CLASS_NAMES[i]: float(probs[i]) for i in range(NB_CLASSES)}

    top = int(np.argmax(probs))
    conf = float(probs[top])
    plant, disease = pretty(CLASS_NAMES[top])
    etat = "Saine ✅" if "healthy" in CLASS_NAMES[top].lower() else "Malade ⚠️"
    maladie = "—" if "healthy" in CLASS_NAMES[top].lower() else (disease or "Maladie détectée")
    alerte = "" if conf >= 0.70 else "\n\n⚠️ Confiance faible : prends une photo plus nette."
    diagnostic = ("### 🌿 Diagnostic\n"
                  "- **Plante :** %s\n- **État :** %s\n- **Maladie :** %s\n- **Confiance :** %.1f%%%s"
                  % (plant, etat, maladie, conf * 100, alerte))
    return confidences, diagnostic

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Photo de la feuille"),
    outputs=[gr.Label(num_top_classes=5, label="Top 5 prédictions"),
             gr.Markdown(label="Diagnostic")],
    title="🌿 Détection de Maladies des Plantes",
    description="Modèle : %s — PlantVillage (%d classes)" % (
        getattr(model, "name", "CNN"), NB_CLASSES),
)
demo.launch(share=True, debug=False)